In [2]:
!pip install -q causal-learn pgmpy


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [3]:
# from google.colab import drive
# drive.mount('/content/drive')

PROJECT_DIR = "data/pgm-final"
RESULTS_DIR = f"{PROJECT_DIR}/results"

import os
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Results dir:", RESULTS_DIR)

Results dir: data/pgm-final/results


In [4]:
# Cell 2: Load prepared data
import numpy as np
import json

X_continuous = np.load(f"{PROJECT_DIR}/activations/X_continuous.npy")
X_binary     = np.load(f"{PROJECT_DIR}/activations/X_binary.npy")
indices_L1   = np.load(f"{PROJECT_DIR}/activations/indices_L1.npy")
indices_L2   = np.load(f"{PROJECT_DIR}/activations/indices_L2.npy")

with open(f"{PROJECT_DIR}/activations/metadata.json", "r") as f:
    metadata = json.load(f)

N_L1 = metadata["n_features_L1"]
N_L2 = metadata["n_features_L2"]
N_TOTAL = metadata["n_features_total"]

print(f"Continuous: {X_continuous.shape}")
print(f"Binary:     {X_binary.shape}")
print(f"L1 features: {N_L1}, L2 features: {N_L2}, total: {N_TOTAL}")

Continuous: (5000, 76)
Binary:     (5000, 76)
L1 features: 30, L2 features: 46, total: 76


In [5]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from pgmpy.estimators import PC
import pandas as pd
import time

/Users/siddharthraj/Documents/my-projects/pgm-final/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
import pickle
import os

PICKLE_PATH = f"{RESULTS_DIR}/pc_fisherz_alpha05_depth3.pkl"
df_continuous = pd.DataFrame(X_continuous)
est = PC(data=df_continuous)

if os.path.exists(PICKLE_PATH):
    print(f"Loading skeleton from {PICKLE_PATH}")
    with open(PICKLE_PATH, "rb") as f:
        pc_result = pickle.load(f)
    skeleton = pc_result["skeleton"]
    sep_sets = pc_result["sep_sets"]
    elapsed = pc_result["time"]
    print(f"  Edges: {skeleton.number_of_edges()}")
    print(f"  Original runtime: {elapsed:.1f}s (skipped recomputation)")
else:
    print(f"No pickle found, running PC...")
    t0 = time.time()
    skeleton, sep_sets = est.build_skeleton(
        ci_test="pearsonr",
        significance_level=0.05,
        max_cond_vars=3,
        show_progress=True,
    )
    elapsed = time.time() - t0
    print(f"  Edges: {skeleton.number_of_edges()}")
    print(f"  Runtime: {elapsed:.1f}s")

    # Save immediately so next session can reload
    pc_result = {
        "skeleton": skeleton,
        "sep_sets": sep_sets,
        "time": elapsed,
        "alpha": 0.05,
        "max_cond_vars": 3,
        "ci_test": "pearsonr",
    }
    with open(PICKLE_PATH, "wb") as f:
        pickle.dump(pc_result, f)
    print(f"  Saved to {PICKLE_PATH}")

Loading skeleton from data/pgm-final/results/pc_fisherz_alpha05_depth3.pkl
  Edges: 382
  Original runtime: 1395.7s (skipped recomputation)


In [7]:
print(f"Number of edges: {skeleton.number_of_edges()}")

Number of edges: 382


In [8]:
import pickle

result = {
    "skeleton": skeleton,
    "sep_sets": sep_sets,
    "time": elapsed,
    "alpha": 0.05,
    "max_cond_vars": 3,
    "ci_test": "pearsonr",
    "n_edges": skeleton.number_of_edges(),
}

with open(f"{RESULTS_DIR}/pc_fisherz_alpha05_depth3.pkl", "wb") as f:
    pickle.dump(result, f)

print("Saved.")
print("File:", f"{RESULTS_DIR}/pc_fisherz_alpha05_depth3.pkl")

Saved.
File: data/pgm-final/results/pc_fisherz_alpha05_depth3.pkl


In [9]:
import numpy as np

# Convert skeleton edges to (i, j) pairs with i < j
edges = list(skeleton.edges())
edges_arr = np.array([(min(e), max(e)) for e in edges])

n_L1_L1 = sum(1 for (i, j) in edges_arr if j < N_L1)
n_L2_L2 = sum(1 for (i, j) in edges_arr if i >= N_L1)
n_L1_L2 = sum(1 for (i, j) in edges_arr if i < N_L1 <= j)

print(f"Total edges: {len(edges_arr)}")
print(f"  L1—L1 (within layer 1):  {n_L1_L1}")
print(f"  L2—L2 (within layer 2):  {n_L2_L2}")
print(f"  L1—L2 (cross-layer):     {n_L1_L2}")
print()
print(f"Max possible L1—L1: {N_L1*(N_L1-1)//2}")
print(f"Max possible L2—L2: {N_L2*(N_L2-1)//2}")
print(f"Max possible L1—L2: {N_L1*N_L2}")

Total edges: 382
  L1—L1 (within layer 1):  64
  L2—L2 (within layer 2):  142
  L1—L2 (cross-layer):     176

Max possible L1—L1: 435
Max possible L2—L2: 1035
Max possible L1—L2: 1380


In [10]:
import pickle

result = {
    "skeleton": skeleton,
    "sep_sets": sep_sets,
    "time": elapsed,
    "alpha": 0.05,
    "max_cond_vars": 3,
    "ci_test": "pearsonr",
    "n_edges_total": skeleton.number_of_edges(),
    "n_L1_L1": 64,
    "n_L2_L2": 142,
    "n_L1_L2": 176,
}

with open(f"{RESULTS_DIR}/pc_fisherz_alpha05_depth3.pkl", "wb") as f:
    pickle.dump(result, f)

print("Saved with edge breakdown.")

Saved with edge breakdown.


In [11]:
temporal_ordering = {}
for i in range(N_L1):
    temporal_ordering[i] = 0
for i in range(N_L1, N_TOTAL):
    temporal_ordering[i] = 1

print("First few entries:", {k: temporal_ordering[k] for k in list(temporal_ordering)[:5]})
print("L1 entries (time=0):", sum(1 for v in temporal_ordering.values() if v == 0))
print("L2 entries (time=1):", sum(1 for v in temporal_ordering.values() if v == 1))

First few entries: {0: 0, 1: 0, 2: 0, 3: 0, 4: 0}
L1 entries (time=0): 30
L2 entries (time=1): 46


In [12]:
import time

t0 = time.time()
pdag = est.orient_colliders(skeleton, sep_sets, temporal_ordering=temporal_ordering)
elapsed_orient = time.time() - t0

print(f"\nOrientation completed in {elapsed_orient:.1f}s")
print(f"PDAG type: {type(pdag)}")
print(f"Attributes: {[a for a in dir(pdag) if not a.startswith('_')]}")


Orientation completed in 0.0s
PDAG type: <class 'pgmpy.base.PDAG.PDAG'>
Attributes: ['add_edge', 'add_edges_from', 'add_node', 'add_nodes_from', 'add_weighted_edges_from', 'adj', 'adjacency', 'adjlist_inner_dict_factory', 'adjlist_outer_dict_factory', 'all_neighbors', 'apply_meeks_rules', 'calibrate_directed_undirected_edges', 'chain_component', 'clear', 'clear_edges', 'copy', 'degree', 'directed_children', 'directed_edges', 'directed_parents', 'edge_attr_dict_factory', 'edge_subgraph', 'edges', 'exposures', 'get_edge_data', 'get_role', 'get_role_dict', 'get_roles', 'graph', 'graph_attr_dict_factory', 'has_acyclic_extension', 'has_directed_edge', 'has_edge', 'has_node', 'has_predecessor', 'has_role', 'has_semidirected_path', 'has_successor', 'has_undirected_edge', 'in_degree', 'in_edges', 'is_adjacent', 'is_clique', 'is_directed', 'is_multigraph', 'is_valid_causal_structure', 'latents', 'name', 'nbunch_iter', 'neighbors', 'node_attr_dict_factory', 'node_dict_factory', 'nodes', 'number

In [13]:
import time

t0 = time.time()
pdag_complete = pdag.apply_meeks_rules()
elapsed_meeks = time.time() - t0

print(f"Meek's rules applied in {elapsed_meeks:.3f}s")
print(f"Result type: {type(pdag_complete)}")
print(f"Total edges: {pdag_complete.number_of_edges()}")
print(f"Directed edges: {len(list(pdag_complete.directed_edges))}")
print(f"Undirected edges: {len(list(pdag_complete.undirected_edges))}")

Meek's rules applied in 0.001s
Result type: <class 'pgmpy.base.PDAG.PDAG'>
Total edges: 181
Directed edges: 181
Undirected edges: 0


In [14]:
print("PDAG (after v-structure orientation, before Meek's):")
print(f"  total: {pdag.number_of_edges()}")
print(f"  directed: {len(list(pdag.directed_edges))}")
print(f"  undirected: {len(list(pdag.undirected_edges))}")
print()
print("PDAG complete (after Meek's):")
print(f"  total: {pdag_complete.number_of_edges()}")
print(f"  directed: {len(list(pdag_complete.directed_edges))}")
print(f"  undirected: {len(list(pdag_complete.undirected_edges))}")
print()
print("Skeleton:")
print(f"  edges: {skeleton.number_of_edges()}")

PDAG (after v-structure orientation, before Meek's):
  total: 181
  directed: 181
  undirected: 0

PDAG complete (after Meek's):
  total: 181
  directed: 181
  undirected: 0

Skeleton:
  edges: 382


In [15]:
skeleton_edges = set(frozenset(e) for e in skeleton.edges())
pdag_directed = set(frozenset((u, v)) for u, v in pdag_complete.directed_edges)
pdag_undirected = set(frozenset((u, v)) for u, v in pdag_complete.undirected_edges)
pdag_all = pdag_directed | pdag_undirected

in_both = skeleton_edges & pdag_all
in_skel_only = skeleton_edges - pdag_all
in_pdag_only = pdag_all - skeleton_edges

print(f"Skeleton edges:           {len(skeleton_edges)}")
print(f"PDAG edges (all):         {len(pdag_all)}")
print(f"In both:                  {len(in_both)}")
print(f"In skeleton only (lost):  {len(in_skel_only)}")
print(f"In PDAG only (new):       {len(in_pdag_only)}")

Skeleton edges:           382
PDAG edges (all):         181
In both:                  181
In skeleton only (lost):  201
In PDAG only (new):       0


In [16]:
pdag_orig_directed = set(frozenset((u, v)) for u, v in pdag.directed_edges)
pdag_orig_undirected = set(frozenset((u, v)) for u, v in pdag.undirected_edges)
pdag_orig_all = pdag_orig_directed | pdag_orig_undirected

print(f"pdag (post-collider) edges (all):     {len(pdag_orig_all)}")
print(f"pdag (post-collider) directed:        {len(pdag_orig_directed)}")
print(f"pdag (post-collider) undirected:      {len(pdag_orig_undirected)}")

pdag (post-collider) edges (all):     181
pdag (post-collider) directed:        181
pdag (post-collider) undirected:      0


In [21]:
from causallearn.utils.PCUtils import UCSepset, Meek
from causallearn.graph.GraphClass import CausalGraph
import inspect

print("UCSepset.uc_sepset:")
print(inspect.signature(UCSepset.uc_sepset))
print()
print("Meek.meek:")
print(inspect.signature(Meek.meek))
print()
print("CausalGraph init:")
print(inspect.signature(CausalGraph.__init__))

UCSepset.uc_sepset:
(cg: 'CausalGraph', priority: 'int' = 3, background_knowledge: 'BackgroundKnowledge | None' = None) -> 'CausalGraph'

Meek.meek:
(cg: 'CausalGraph', background_knowledge: 'BackgroundKnowledge | None' = None) -> 'CausalGraph'

CausalGraph init:
(self, no_of_var: 'int', node_names: 'List[str] | None' = None)


In [22]:
from causallearn.graph.GraphClass import CausalGraph
import numpy as np

cg = CausalGraph(no_of_var=N_TOTAL)

# Start with no edges (set the entire graph matrix to 0)
cg.G.graph = np.zeros((N_TOTAL, N_TOTAL), dtype=np.int32)

# Add each skeleton edge as undirected: both entries = -1
for (i, j) in skeleton.edges():
    i, j = int(i), int(j)
    cg.G.graph[i, j] = -1
    cg.G.graph[j, i] = -1

print(f"Edges in cg (undirected): {(cg.G.graph == -1).sum() // 2}")
print(f"Expected from skeleton:   {skeleton.number_of_edges()}")

Edges in cg (undirected): 382
Expected from skeleton:   382
